# ECG per-pathology fine-tune — lift macro balanced-accuracy & weak-class F1 (GPU)

**Goal.** Continue-train each of the 7 ecglib DenseNet-1D pathology classifiers,
starting from their **pretrained** weights, to push **macro balanced-accuracy
0.884 → ≥ 0.90** and raise the weak-class F1s (SBRAD, 1AVB) — *without* hurting
the strong classes (AFIB, RBBB, LBBB, PVC).

**Baseline (measured in this repo).** `tools/eval_ecg_classifier.py` on PTB-XL
**fold 10** (official held-out test, 2,198 records; thresholds tuned on fold 9):
- **Mean ROC-AUC 0.978**
- **Macro balanced accuracy 0.884**  (0.50 = trivial all-negative model)
- **Macro F1 0.71** — dragged down by SBRAD (F1 ≈ 0.47) and 1AVB (F1 ≈ 0.52)

**Realistic target — read this before your defence.**
- Aim: **macro balanced-acc ≥ 0.90** and **better weak-class F1**.
- This is **NOT** a promise of 95% on every pathology. SBRAD/1AVB are precision-
  limited (many false positives) and will not reach the AFIB/RBBB level on PTB-XL.
- **AUC is already near-ceiling (0.978).** These models are excellent *rankers*;
  fine-tuning mostly sharpens the decision boundary, so expect AUC to stay roughly
  flat while balanced-acc / F1 move. A big AUC jump would be a leakage red flag,
  not good news.

**No-regression rule (coded into the loop).** A fine-tuned `<PATHOLOGY>.pt` is
saved **only if** it beats the pretrained baseline on fold 10:
`balanced-acc(ft) > balanced-acc(base)` **AND** `AUC(ft) ≥ AUC(base) − 0.005`.
Otherwise the cell prints **`kept baseline`** and writes nothing for that
pathology — so the deployed ensemble can never regress.

**How the local code picks up the new weights.**
`model_loader.get_ecg_models()` was patched to load
`backend/models_weights/ecg_finetuned/<PATHOLOGY>.pt` (plain `state_dict`) over
the ecglib weights when present (override dir via `ECG_FINETUNED_DIR`). Drop the
kept `.pt` files there — no code change, and a missing file = stock ecglib.

**Runtime.** 7 models × a few epochs on a **T4** ≈ **1–3 h**, plus a one-time
~10–15 min PTB-XL preprocessing pass. To run a subset, edit the `PATHOLOGIES`
list in the config cell (e.g. just the weak classes `["SBRAD", "1AVB"]`).

Run the cells **top to bottom**. The final cell prints a `=== RESULTS ===` block —
paste it into `Colab PFE/RESULTS_TEMPLATE.md` and hand it back to Claude.

## 1 — Confirm the GPU
If this prints `NO GPU`, set **Runtime → Change runtime type → T4 GPU** and re-run.
Training 7 DenseNet-1D models is impractically slow on CPU.

In [ ]:
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 2 — Mount Drive and unpack the repo
Same pattern as `tools/COLAB.md`: unzip `My Drive/medical-platform.zip` to
`/content/medical-platform`. We need the repo for the **exact** eval methodology
(`tools/eval_ecg_classifier.py`: PTB-XL adapter, label mapping, AUC) and the
preprocessing (`backend/apps/inference/utils.load_ecg_signal`). Build the zip
first with `python "Colab PFE/make_lean_zip.py"` and upload it to your Drive root.
The flatten check handles a zip that nested an extra top-level folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/medical-platform
!mkdir -p /content/medical-platform
!unzip -q "/content/drive/My Drive/medical-platform.zip" -d /content/medical-platform
import os
root = "/content/medical-platform"
if not os.path.exists(f"{root}/tools") and os.path.exists(f"{root}/medical-platform/tools"):
    root = f"{root}/medical-platform"
print("repo root:", root)
%cd $root

## 3 — Install the extra deps (+ Python 3.12 patch)
Colab already ships torch / numpy / pandas / scipy / matplotlib. We add the exact
**ecglib 1.0.1** pin (the repo's hard constraint — other versions change the
`create_model` import path) plus `wfdb` (PTB-XL is in WFDB `.dat/.hea` format).
ecglib declares `torch`/`wfdb` **unpinned**, so this install keeps Colab's GPU
torch untouched.

**Python 3.12 note.** Colab now runs Python 3.12, which removed the legacy `imp`
module — and `ecglib 1.0.1` contains one stray `import imp` it never actually
uses (verified: zero `imp.` calls in the whole package). The cell below comments
out that single line after installing, which makes ecglib import cleanly. The
project's *local* install is unaffected (it pins Python 3.10/3.11, where `imp`
still exists). If you restart the runtime, just re-run this cell — it re-installs
and re-patches.

In [ ]:
!pip -q install ecglib==1.0.1 wfdb

# --- Python 3.12 compatibility patch ------------------------------------
# ecglib 1.0.1 has a stray `import imp` in models/config/model_configs.py.
# `imp` was removed in Python 3.12 (Colab's default) and ecglib never uses
# it, so we comment out that one line. find_spec locates the file WITHOUT
# importing the package (importing is what crashes).
import importlib.util
import pathlib
import re

spec = importlib.util.find_spec("ecglib")
cfg = pathlib.Path(spec.origin).parent / "models" / "config" / "model_configs.py"
src = cfg.read_text()
if re.search(r"^import imp\b", src, flags=re.M):
    src = re.sub(r"^import imp\b.*$",
                 "# import imp  (removed in Py3.12; ecglib never uses it)",
                 src, count=1, flags=re.M)
    cfg.write_text(src)
    print("patched for Python 3.12:", cfg)
else:
    print("no patch needed (already patched):", cfg)

import wfdb, ecglib
print("ecglib", ecglib.__version__ if hasattr(ecglib, "__version__") else "(installed)", "| wfdb", wfdb.__version__)

## 4 — Run configuration (edit here)
`PATHOLOGIES` is the main knob: trim it to run a subset (e.g. only the weak
classes) to fit a session. The 7-model default is the full run (≈ 1–3 h on a T4).

The folds and preprocessing below are **fixed to match the eval harness** — do
not change them or the baseline comparison becomes apples-to-oranges:
- **train = strat_folds 1–8**, **validation = fold 9**, **test = fold 10**
- PTB-XL `filename_hr` (**500 Hz**, `records500/`), 12-lead, 10 s = (12, 5000)
- **0.5–40 Hz** 4th-order band-pass (`filtfilt`) + **per-lead z-score**
- label = SCP code present with likelihood ≥ `MIN_LIKELIHOOD` (0 = key-present,
  the PTB-XL benchmark convention), mapped via `SCP_TO_ECGLIB`

In [ ]:
import time

# --- main knob: trim to run a subset ---
PATHOLOGIES = ["AFIB", "1AVB", "STACH", "SBRAD", "RBBB", "LBBB", "PVC"]   # full 7-model run
# e.g. weak classes only:  PATHOLOGIES = ["SBRAD", "1AVB"]

# --- training hyper-parameters ---
EPOCHS     = 8        # few epochs: weights are already pretrained
BATCH_SIZE = 64
LR         = 1e-4     # small LR so we fine-tune, not overwrite, the pretraining
PATIENCE   = 3        # early-stop on val balanced-acc
SEED       = 0

# --- fixed eval methodology (match tools/eval_ecg_classifier.py) ---
TRAIN_FOLDS    = [1, 2, 3, 4, 5, 6, 7, 8]
VAL_FOLD       = 9
TEST_FOLD      = 10
MIN_LIKELIHOOD = 0.0   # PTB-XL: key-present counts as positive (benchmark convention)
AUC_TOLERANCE  = 0.005 # no-regression: AUC may not drop by more than this

# --- paths ---
PTBXL_DIR  = "physionet.org/files/ptb-xl/1.0.3"          # set by the download cell
OUT_DIR    = "outputs/ecg_finetuned"                      # local kept checkpoints
DRIVE_OUT  = "/content/drive/My Drive/colab_pfe_outputs/ecg_finetuned"

import numpy as np
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
T_START = time.time()
os.makedirs(OUT_DIR, exist_ok=True)
print("config:", dict(PATHOLOGIES=PATHOLOGIES, EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE,
                      LR=LR, PATIENCE=PATIENCE, SEED=SEED))

## 5 — Download PTB-XL from PhysioNet (official zip — one connection)
An earlier version of this cell used recursive `wget` over the file tree —
that issues ~43,000 individual requests (one per record file) and PhysioNet
throttles the pattern so hard the cell can sit silent for an hour. The official
**single zip** (~1.7 GB, one connection) is the reliable path: visible progress
bar, resumable (`-c`), ~5–15 min. After unzipping we move the data into the
`physionet.org/files/ptb-xl/1.0.3` layout the rest of the notebook expects and
drop the unused 100 Hz copies. If a previous stalled run left a partial tree,
the cell detects it (file count too low), wipes it, and re-downloads cleanly.

In [ ]:
# PTB-XL — three sources, fastest first:
#   1) already extracted on this VM           -> skip everything
#   2) ptbxl.zip uploaded to your Drive root  -> copy (~2-3 min), no internet
#   3) official PhysioNet single zip          -> download (one connection;
#      recursive wget ~43k requests gets throttled and stalls — never use it)
import os, shutil

ZIP_URL = ("https://physionet.org/static/published-projects/ptb-xl/"
           "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip")
DRIVE_ZIPS = ["/content/drive/My Drive/ptbxl.zip",
              "/content/drive/My Drive/ptb-xl-a-large-publicly-available-"
              "electrocardiography-dataset-1.0.3.zip"]
ZIP_PATH    = "/content/ptbxl.zip"
EXTRACT_DIR = "/content/ptbxl_extract"

def ptbxl_complete():
    if not os.path.exists(f"{PTBXL_DIR}/ptbxl_database.csv"):
        return False
    n = sum(len(fs) for _, _, fs in os.walk(f"{PTBXL_DIR}/records500"))
    return n >= 43000        # full set = 21,799 records x 2 files (.dat + .hea)

if not ptbxl_complete():
    shutil.rmtree(PTBXL_DIR, ignore_errors=True)   # partial tree from an old stall
    if not os.path.exists(ZIP_PATH):
        drive_zip = next((z for z in DRIVE_ZIPS if os.path.exists(z)), None)
        if drive_zip:
            print("copying the zip from Drive (no download needed, ~2-3 min)...")
            shutil.copy(drive_zip, ZIP_PATH)
        else:
            !wget -c --progress=bar:force:noscroll "{ZIP_URL}" -O "{ZIP_PATH}"
    print("unzipping (~2 min, silent)...")
    !unzip -q -o "{ZIP_PATH}" -d "{EXTRACT_DIR}"
    src = f"{EXTRACT_DIR}/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
    if not os.path.exists(f"{src}/ptbxl_database.csv"):   # zip with a different top folder
        src = next(d for d, _, fs in os.walk(EXTRACT_DIR) if "ptbxl_database.csv" in fs)
    os.makedirs(os.path.dirname(PTBXL_DIR), exist_ok=True)
    shutil.move(src, PTBXL_DIR)
    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
    os.remove(ZIP_PATH)                                            # free 1.7 GB
    shutil.rmtree(f"{PTBXL_DIR}/records100", ignore_errors=True)   # 100 Hz copies unused

assert ptbxl_complete(), f"PTB-XL incomplete at {PTBXL_DIR} — re-run this cell"
n500 = sum(len(fs) for _, _, fs in os.walk(f"{PTBXL_DIR}/records500"))
print("PTB-XL ready at", PTBXL_DIR, "| files under records500/:", n500)

## 6 — Build the fold tensors once (RAM-safe, crash-proof)
Same faithful preprocessing as the repo (0.5–40 Hz band-pass + per-lead z-score
on the repo's own `iter_ptbxl` / `load_ecg_signal`), with two robustness fixes
learned the hard way:

- **Arrays live on disk, not RAM.** The signals (~2.6 GB float16) are written to
  memory-mapped `.npy` files under `/content/` — system RAM stays low, so the
  "session crashed after using all available RAM" failure can't recur. Batches
  are read from disk and cast to `float32` on the GPU per step (negligible cost).
- **Preprocessing is cached.** If the `.npy` files already exist (e.g. after a
  kernel crash or restart), they are reused instantly instead of redoing the
  ~10–15 min pass. Delete `/content/*_X.npy` to force a rebuild.

The cell also re-anchors cwd at the repo root and re-locates PTB-XL, so it
survives kernel restarts no matter what order things died in.

In [ ]:
import os, sys, time, gc
from pathlib import Path
from scipy.signal import butter, filtfilt

# --- restart-proof anchoring --------------------------------------------
# A kernel restart resets cwd to /content (the %cd from section 2 is lost).
# Re-anchor at the repo root and re-locate PTB-XL wherever the download
# cell actually put it (it extracts relative to whatever cwd was then).
root = "/content/medical-platform"
if not os.path.exists(f"{root}/backend/apps") and \
        os.path.exists(f"{root}/medical-platform/backend/apps"):
    root = f"{root}/medical-platform"
assert os.path.exists(f"{root}/backend/apps"), \
    "repo not found — re-run the section-2 cell (mount Drive + unzip), then re-run this one"
os.chdir(root)
for _c in (PTBXL_DIR,
           f"{root}/physionet.org/files/ptb-xl/1.0.3",
           "/content/physionet.org/files/ptb-xl/1.0.3"):
    if os.path.exists(f"{_c}/ptbxl_database.csv"):
        PTBXL_DIR = _c
        break
else:
    raise AssertionError("PTB-XL not found — re-run the section-5 download cell")
print("repo root:", root, "| PTB-XL:", PTBXL_DIR)

sys.path.insert(0, f"{root}/backend")
sys.path.insert(0, f"{root}/tools")
from apps.inference.utils import load_ecg_signal
from eval_ecg_classifier import iter_ptbxl, SCP_TO_ECGLIB, PATHOLOGIES as ALL_PATHOLOGIES, auc_score

assert all(p in ALL_PATHOLOGIES for p in PATHOLOGIES), "unknown pathology in PATHOLOGIES"
P_IDX = {p: ALL_PATHOLOGIES.index(p) for p in ALL_PATHOLOGIES}  # column order = repo order

def _preprocess(path):
    # load_ecg_signal returns 3 values (signal, fs, lead_quality). The 3rd is
    # unused here but MUST be unpacked — a 2-value unpack raises "too many values
    # to unpack" on EVERY record, which build_split silently skips, producing an
    # empty split and all-zero / NaN metrics. (This was the all-"no positives" bug.)
    signal, fs, _ = load_ecg_signal(str(path))                   # (12, 5000) @ 500 Hz
    b, a = butter(4, [0.5, 40], btype="bandpass", fs=fs)
    filtered = filtfilt(b, a, signal, axis=1)
    norm = (filtered - filtered.mean(axis=1, keepdims=True)) / \
           (filtered.std(axis=1, keepdims=True) + 1e-8)
    return norm.astype(np.float16)

def build_split(folds, name):
    """Disk-backed (RAM-safe) split builder with a crash-proof cache."""
    xf, yf = f"/content/{name}_X.npy", f"/content/{name}_Y.npy"
    if os.path.exists(xf) and os.path.exists(yf):
        Y = np.load(yf)
        X = np.load(xf, mmap_mode="r")[:len(Y)]
        print(f"{name}: reusing cached preprocessed arrays ({len(Y)} records)")
        return X, Y
    items = []
    for f in folds:
        items.extend(iter_ptbxl(Path(PTBXL_DIR), MIN_LIKELIHOOD, f))
    n = len(items)
    if n == 0:
        raise RuntimeError(f"{name}: iter_ptbxl yielded 0 records — PTB-XL missing/incomplete")
    # memmap on disk, NOT a RAM array — this is what prevents the OOM crash
    X = np.lib.format.open_memmap(xf + ".tmp", mode="w+",
                                  dtype=np.float16, shape=(n, 12, 5000))
    Y = np.zeros((n, len(ALL_PATHOLOGIES)), dtype=np.int8)
    ok, n_fail, first_err = 0, 0, None
    for path, positives in items:
        try:
            X[ok] = _preprocess(path)
        except Exception as e:
            n_fail += 1
            if first_err is None:
                first_err = f"{type(e).__name__}: {e}"
            continue                                              # skip unreadable record
        for c in positives:
            Y[ok, ALL_PATHOLOGIES.index(c)] = 1
        ok += 1
        if ok % 1000 == 0:
            print(f"  {name}: {ok}/{n}")
            gc.collect()
    X.flush(); del X
    # Fail LOUDLY rather than caching an empty / label-less split (the old bug).
    if ok == 0:
        raise RuntimeError(f"{name}: 0/{n} records preprocessed ({n_fail} failed). "
                           f"First error -> {first_err}")
    if int(Y[:ok].sum()) == 0:
        raise RuntimeError(f"{name}: {ok} records but ZERO positive labels — SCP mapping broken")
    if n_fail:
        print(f"  {name}: WARNING {n_fail}/{n} failed (first: {first_err})")
    os.replace(xf + ".tmp", xf)        # cache becomes valid only when complete
    np.save(yf, Y[:ok])
    print(f"{name}: {ok} records, {int(Y[:ok].sum())} positive labels")
    return np.load(xf, mmap_mode="r")[:ok], np.load(yf)

t_pp = time.time()
Xtr, Ytr = build_split(TRAIN_FOLDS, "train")
Xval, Yval = build_split([VAL_FOLD], "val")
Xte, Yte = build_split([TEST_FOLD], "test")
print("preprocessing wall-clock: {:.0f} min".format((time.time() - t_pp) / 60))
print("class positives (test):", {p: int(Yte[:, P_IDX[p]].sum()) for p in PATHOLOGIES})

## 7 — Train / eval helpers
`logit()` extracts the raw (B,) logit from the ecglib head (output is (B, 1)).
Probabilities use the same sigmoid read-out as deployment (`_scalar_probability`).
Balanced-accuracy thresholds are tuned **on fold 9 only** (no test leakage) — the
same fold the repo tunes on; here we optimise them for **balanced accuracy**
rather than F1, so the baseline macro balanced-acc lands in the ~0.88 ballpark of
`VALIDATION.md`. AUC is the repo's dependency-free Mann–Whitney `auc_score`
(threshold-independent).

In [ ]:
import torch.nn as nn
import copy
device = "cuda" if torch.cuda.is_available() else "cpu"

def logit(model, xb):
    out = model(xb)
    if isinstance(out, tuple):
        out = out[0]
    return out.view(out.size(0), -1)[:, 0]            # (B,) raw logits

@torch.no_grad()
def predict_probs(model, X, bs=128):
    model.eval()
    out = np.zeros(len(X), dtype=np.float32)
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs].astype(np.float32)).to(device)
        out[i:i + xb.size(0)] = torch.sigmoid(logit(model, xb)).cpu().numpy()
    return out

def bal_acc(probs, y, thr):
    pred = (probs >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return 0.5 * (sens + spec)

def f1_at(probs, y, thr):
    pred = (probs >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

def best_thr_balacc(probs, y):
    best_t, best = 0.5, -1.0
    if y.sum() == 0 or y.sum() == len(y):
        return 0.5
    for t in np.linspace(0.01, 0.99, 99):
        b = bal_acc(probs, y, t)
        if b > best:
            best, best_t = b, float(t)
    return best_t

def evaluate(model, j):
    """Tune the balanced-acc threshold on fold 9, then score fold 10."""
    pv, pt = predict_probs(model, Xval), predict_probs(model, Xte)
    yv, yt = Yval[:, j], Yte[:, j]
    thr = best_thr_balacc(pv, yv)
    auc = auc_score(pt, yt)
    return {"thr": thr,
            "bal_acc": bal_acc(pt, yt, thr),
            "f1": f1_at(pt, yt, thr),
            "auc": float(auc) if auc is not None else float("nan")}

def val_balacc(model, j):
    pv = predict_probs(model, Xval)
    yv = Yval[:, j]
    return bal_acc(pv, yv, best_thr_balacc(pv, yv))

## 8 — Per-pathology fine-tune loop (with the no-regression rule)
For each pathology: build the model from **ecglib pretrained** weights, record the
baseline fold-10 metrics, then fine-tune on folds 1–8 with a class-balanced
`BCEWithLogitsLoss` (`pos_weight = n_neg/n_pos`), early-stopping on fold-9 balanced
accuracy. The best-val state is re-scored on fold 10 and **kept only if it passes
the no-regression rule**; otherwise we keep the baseline (write nothing).

In [ ]:
from ecglib.models import create_model

results = []   # one dict per pathology

for p in PATHOLOGIES:
    j = ALL_PATHOLOGIES.index(p)
    print(f"\n{'=' * 64}\n{p}\n{'=' * 64}")
    model = create_model(model_name="densenet1d121", pathology=p, pretrained=True).to(device)

    base = evaluate(model, j)
    print(f"  baseline (fold10): bal_acc={base['bal_acc']:.3f}  AUC={base['auc']:.3f}  "
          f"F1={base['f1']:.3f}  thr={base['thr']:.2f}")

    y_tr = Ytr[:, j].astype(np.float32)
    n_pos = int(y_tr.sum()); n_neg = len(y_tr) - n_pos
    if n_pos == 0:
        print("  no positive training samples — kept baseline")
        results.append({"p": p, **{f"base_{k}": base[k] for k in ("bal_acc", "auc", "f1")},
                        "ft_bal_acc": base["bal_acc"], "ft_auc": base["auc"], "ft_f1": base["f1"],
                        "kept": False, "reason": "no positives"})
        del model; torch.cuda.empty_cache(); continue

    pos_weight = torch.tensor([max(n_neg / max(n_pos, 1), 1.0)], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    idx = np.arange(len(Xtr))

    best_val, best_state, bad = -1.0, None, 0
    for ep in range(EPOCHS):
        model.train()
        rng.shuffle(idx)
        running = 0.0
        for i in range(0, len(idx), BATCH_SIZE):
            bi = idx[i:i + BATCH_SIZE]
            xb = torch.from_numpy(Xtr[bi].astype(np.float32)).to(device)
            yb = torch.from_numpy(y_tr[bi]).to(device)
            opt.zero_grad()
            loss = crit(logit(model, xb), yb)
            loss.backward(); opt.step()
            running += loss.item() * len(bi)
        vba = val_balacc(model, j)
        print(f"  epoch {ep + 1}/{EPOCHS}  train_loss={running / len(idx):.4f}  val_bal_acc={vba:.3f}")
        if vba > best_val:
            best_val = vba
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"  early stop (no val improvement in {PATIENCE} epochs)")
                break

    model.load_state_dict(best_state)
    ft = evaluate(model, j)
    print(f"  fine-tuned (fold10): bal_acc={ft['bal_acc']:.3f}  AUC={ft['auc']:.3f}  "
          f"F1={ft['f1']:.3f}  thr={ft['thr']:.2f}")

    kept = (ft["bal_acc"] > base["bal_acc"]) and (ft["auc"] >= base["auc"] - AUC_TOLERANCE)
    if kept:
        torch.save(best_state, os.path.join(OUT_DIR, f"{p}.pt"))
        print(f"  SAVED {p}.pt  (bal_acc {base['bal_acc']:.3f}->{ft['bal_acc']:.3f}, "
              f"AUC {base['auc']:.3f}->{ft['auc']:.3f})")
        reason = "passed no-regression rule"
    else:
        why = "bal_acc not improved" if ft["bal_acc"] <= base["bal_acc"] else "AUC dropped > 0.005"
        print(f"  kept baseline  ({why})")
        reason = why

    results.append({"p": p,
                    "base_bal_acc": base["bal_acc"], "base_auc": base["auc"], "base_f1": base["f1"],
                    "ft_bal_acc": ft["bal_acc"], "ft_auc": ft["auc"], "ft_f1": ft["f1"],
                    "kept": bool(kept), "reason": reason})
    del model; torch.cuda.empty_cache()

print("\nloop done.")

## 9 — Save kept checkpoints + metrics JSON to Drive
Copies the `<PATHOLOGY>.pt` files that passed the no-regression rule, plus an
`ecg_finetune_metrics.json` (per-pathology before/after), to
`My Drive/colab_pfe_outputs/ecg_finetuned/`. Place the `.pt` files locally at
`backend/models_weights/ecg_finetuned/` — `get_ecg_models()` auto-detects them.

In [ ]:
import json, shutil

# if a restart unmounted Drive, remount — otherwise makedirs would create a
# fake local /content/drive/... folder and the "saved" files would die with the VM
if not os.path.isdir("/content/drive/My Drive"):
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(DRIVE_OUT, exist_ok=True)

def _macro(key_kept, key_base):
    """Macro over PATHOLOGIES using the DEPLOYED weights (ft if kept else baseline)."""
    vals = [(r[key_kept] if r["kept"] else r[key_base]) for r in results]
    return float(np.mean(vals)) if vals else 0.0

summary = {
    "run": "ECG per-pathology fine-tune (ecglib DenseNet-1D, PTB-XL fold 10)",
    "baseline_reference": {"mean_auc": 0.978, "macro_balanced_acc": 0.884, "macro_f1": 0.71},
    "pathologies": PATHOLOGIES,
    "config": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR,
               "patience": PATIENCE, "seed": SEED, "min_likelihood": MIN_LIKELIHOOD,
               "auc_tolerance": AUC_TOLERANCE,
               "train_folds": TRAIN_FOLDS, "val_fold": VAL_FOLD, "test_fold": TEST_FOLD},
    "macro_balanced_acc_baseline": float(np.mean([r["base_bal_acc"] for r in results])),
    "macro_balanced_acc_deployed": _macro("ft_bal_acc", "base_bal_acc"),
    "macro_f1_baseline": float(np.mean([r["base_f1"] for r in results])),
    "macro_f1_deployed": _macro("ft_f1", "base_f1"),
    "macro_auc_deployed": _macro("ft_auc", "base_auc"),
    "kept": [r["p"] for r in results if r["kept"]],
    "per_pathology": {r["p"]: {k: (round(v, 4) if isinstance(v, float) else v)
                               for k, v in r.items() if k != "p"} for r in results},
}

with open(os.path.join(DRIVE_OUT, "ecg_finetune_metrics.json"), "w") as f:
    json.dump(summary, f, indent=2)

n_copied = 0
for r in results:
    if r["kept"]:
        src = os.path.join(OUT_DIR, f"{r['p']}.pt")
        if os.path.exists(src):
            shutil.copy(src, os.path.join(DRIVE_OUT, f"{r['p']}.pt"))
            n_copied += 1
print(f"copied {n_copied} kept checkpoint(s) + metrics json to {DRIVE_OUT}")

## 10 — RESULTS (paste this whole block back to Claude)

In [ ]:
print("=== RESULTS - paste this back to Claude ===")
print("run: ECG per-pathology fine-tune (ecglib DenseNet-1D, PTB-XL fold 10)")
print("baseline ref (VALIDATION.md): mean AUC 0.978 | macro balanced-acc 0.884 | macro F1 0.71")
print("note: AUC is near-ceiling; targeting macro balanced-acc >= 0.90 + better weak-class F1")
print("config: epochs={} batch={} lr={} patience={} seed={}".format(
    EPOCHS, BATCH_SIZE, LR, PATIENCE, SEED))
print("pathologies run: {}".format(PATHOLOGIES))
print("")
hdr = "{:<7}{:>10}{:>9}{:>9}{:>9}{:>8}{:>8}{:>8}  {}".format(
    "path", "base_bA", "ft_bA", "base_AUC", "ft_AUC", "base_F1", "ft_F1", "thr", "verdict")
print(hdr); print("-" * len(hdr))
for r in results:
    verdict = "SAVED" if r["kept"] else "kept baseline"
    print("{:<7}{:>10.3f}{:>9.3f}{:>9.3f}{:>9.3f}{:>8.3f}{:>8.3f}{:>8}  {}".format(
        r["p"], r["base_bal_acc"], r["ft_bal_acc"], r["base_auc"], r["ft_auc"],
        r["base_f1"], r["ft_f1"], "", verdict))
print("")
mb = float(np.mean([r["base_bal_acc"] for r in results]))
md_ = summary["macro_balanced_acc_deployed"]
fb = float(np.mean([r["base_f1"] for r in results]))
fd = summary["macro_f1_deployed"]
print("macro balanced-acc : {:.3f} (baseline) -> {:.3f} (deployed)  [target >= 0.90]".format(mb, md_))
print("macro F1           : {:.3f} (baseline) -> {:.3f} (deployed)".format(fb, fd))
print("macro AUC          : {:.3f} (deployed; near-ceiling, expected ~flat)".format(
    summary["macro_auc_deployed"]))
print("kept (beat baseline): {}".format(summary["kept"] or "none"))
print("")
print("files on Drive (My Drive/colab_pfe_outputs/ecg_finetuned/):")
for p in summary["kept"]:
    print("  {}.pt".format(p))
print("  ecg_finetune_metrics.json")
print("place kept <PATHOLOGY>.pt at backend/models_weights/ecg_finetuned/ locally")
print("(get_ecg_models() auto-detects them; ECG_FINETUNED_DIR overrides the dir)")
print("runtime: notebook total {:.0f} min".format((time.time() - T_START) / 60))
print("=== END RESULTS ===")